# 🚀 03 – Visualisations avancées avec Matplotlib

---

## 🎯 Objectifs
- Créer des **subplots complexes** avec des tailles différentes (`gridspec`)
- Utiliser un **double axe Y** pour comparer deux variables d'unités différentes
- Tracer des **barres empilées** — répartition d'un total par catégorie
- Créer un **graphique en aires** — évolution cumulée
- Produire un **tableau de bord complet** prêt pour la présentation

> ℹ️ Ces graphiques sont ceux qu'on retrouve dans les **rapports d'entreprise** et les **tableaux de bord**. Ils combinent les notions des notebooks 00, 01 et 02.

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
print("Matplotlib prêt ✅")

---
## 📝 Partie 1 – Subplots avec tailles différentes (`GridSpec`)

### 🔎 Le problème avec `plt.subplots()`

`plt.subplots(2, 2)` crée une grille régulière où chaque panneau a la même taille. Mais dans un vrai rapport, on veut souvent un **grand graphique principal** et des **petits graphiques secondaires**.

**Exemple de layout professionnel :**
```
┌────────────────────┬────────┐
│                    │  Mini  │
│   Grand graphique  ├────────┤
│   (occupe 2 cols)  │  Mini  │
└────────────────────┴────────┘
```

`GridSpec` permet de définir précisément combien de lignes et colonnes occupe chaque panneau :

```python
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 3)   # grille 2 lignes × 3 colonnes

# Grand panneau : occupe toute la ligne 0 et les 2 premières colonnes
ax_grand  = fig.add_subplot(gs[0, 0:2])   # ligne 0, colonnes 0 et 1

# Petit panneau haut-droit
ax_mini1  = fig.add_subplot(gs[0, 2])     # ligne 0, colonne 2

# Panneau bas qui s'étend sur toute la largeur
ax_bas    = fig.add_subplot(gs[1, :])     # ligne 1, toutes les colonnes
```

**La syntaxe `gs[ligne, colonne]` fonctionne comme le slicing NumPy :**
- `gs[0, 0]` → panneau en ligne 0, colonne 0
- `gs[0, 0:2]` → panneau en ligne 0, colonnes 0 et 1 (2 colonnes)
- `gs[1, :]` → panneau en ligne 1, toutes les colonnes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Données
mois    = ["Jan","Fév","Mar","Avr","Mai","Jun","Jul","Aoû","Sep","Oct","Nov","Déc"]
ca      = [42, 38, 55, 60, 72, 85, 90, 78, 65, 70, 88, 95]
produits= ["Pizza","Burger","Sushi","Tacos"]
ventes  = [320, 280, 180, 150]
regions = ["Nord","Sud","Est","Ouest"]
parts   = [35, 28, 22, 15]

# Layout : 2 lignes × 3 colonnes
fig = plt.figure(figsize=(15, 9))
gs  = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)
#                               ↑ espace entre lignes  ↑ espace entre colonnes

# Grand graphique : ligne 0, colonnes 0 et 1
ax_ca = fig.add_subplot(gs[0, 0:2])
ax_ca.plot(mois, ca, color="steelblue", marker="o", linewidth=2)
ax_ca.axhline(np.mean(ca), color="red", linestyle="--", alpha=0.7,
              label=f"Moyenne : {np.mean(ca):.0f} k€")
ax_ca.set_title("CA mensuel (graphique principal)", fontsize=12)
ax_ca.set_ylabel("CA (k€)")
ax_ca.legend(fontsize=9)
ax_ca.tick_params(axis="x", rotation=45)
ax_ca.grid(True, alpha=0.3)

# Petit graphique haut-droit : camembert
ax_pie = fig.add_subplot(gs[0, 2])
ax_pie.pie(parts, labels=regions, autopct="%1.0f%%", startangle=90,
           colors=["steelblue","coral","seagreen","gold"])
ax_pie.set_title("CA par région", fontsize=11)

# Bas gauche : barres
ax_bar = fig.add_subplot(gs[1, 0:2])
ax_bar.bar(produits, ventes, color=["steelblue","coral","seagreen","gold"])
ax_bar.set_title("Ventes par produit", fontsize=11)
ax_bar.set_ylabel("Quantité")
ax_bar.grid(True, axis="y", alpha=0.3)
for i, v in enumerate(ventes):
    ax_bar.text(i, v + 3, str(v), ha="center", fontsize=9)

# Bas droit : texte récapitulatif
ax_txt = fig.add_subplot(gs[1, 2])
ax_txt.axis("off")   # pas de graphique, juste du texte
ax_txt.text(0.5, 0.7, f"CA Total : {sum(ca)} k€",
            ha="center", fontsize=13, fontweight="bold", color="steelblue")
ax_txt.text(0.5, 0.5, f"Meilleur mois : {mois[ca.index(max(ca))]}",
            ha="center", fontsize=11)
ax_txt.text(0.5, 0.3, f"Mois sous obj : {sum(1 for v in ca if v < 70)}",
            ha="center", fontsize=11, color="crimson")

fig.suptitle("Tableau de bord Commercial — Année en cours",
             fontsize=14, fontweight="bold", y=1.01)
plt.show()

### 🧩 Exercice 1 – Layout GridSpec personnalisé

Créez un layout **3 lignes × 2 colonnes** avec :
- **Ligne 0, toute la largeur** : courbe du CA mensuel sur 12 mois
- **Ligne 1, colonne 0** : histogramme des ventes journalières
- **Ligne 1, colonne 1** : barres des ventes par produit
- **Ligne 2, toute la largeur** : nuage de points (budget pub vs CA)

```python
mois     = ["Jan","Fév","Mar","Avr","Mai","Jun",
            "Jul","Aoû","Sep","Oct","Nov","Déc"]
ca_mois  = [42,38,55,60,72,85,90,78,65,70,88,95]
np.random.seed(5)
ventes_j = np.random.normal(150, 40, 100).clip(50, 300)
produits = ["Pizza","Burger","Sushi","Tacos"]
ventes_p = [320, 280, 180, 150]
budget   = np.random.randint(5, 50, 30)
ca_pub   = budget * 8 + np.random.normal(0, 30, 30)
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

mois     = ["Jan","Fév","Mar","Avr","Mai","Jun",
            "Jul","Aoû","Sep","Oct","Nov","Déc"]
ca_mois  = [42,38,55,60,72,85,90,78,65,70,88,95]
np.random.seed(5)
ventes_j = np.random.normal(150, 40, 100).clip(50, 300)
produits = ["Pizza","Burger","Sushi","Tacos"]
ventes_p = [320, 280, 180, 150]
budget   = np.random.randint(5, 50, 30)
ca_pub   = budget * 8 + np.random.normal(0, 30, 30)

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 2, hspace=0.5, wspace=0.35)

# TODO : Ligne 0 — courbe CA (toute la largeur)
ax0 = fig.add_subplot(gs[0, :])
ax0.plot(mois, ca_mois, color=..., marker="o", linewidth=2)
ax0.set_title("CA mensuel")
ax0.set_ylabel("k€")
ax0.grid(True, alpha=0.3)

# TODO : Ligne 1, col 0 — histogramme ventes journalières
ax1 = fig.add_subplot(gs[1, 0])
ax1.hist(ventes_j, bins=..., color=..., edgecolor="white")
ax1.set_title("Distribution ventes journalières")
ax1.set_xlabel("Ventes (€)")

# TODO : Ligne 1, col 1 — barres produits
ax2 = fig.add_subplot(gs[1, 1])
ax2.bar(produits, ventes_p, color=...)
ax2.set_title("Ventes par produit")
ax2.grid(True, axis="y", alpha=0.3)

# TODO : Ligne 2 — scatter budget vs CA (toute la largeur)
ax3 = fig.add_subplot(gs[2, :])
ax3.scatter(budget, ca_pub, color=..., alpha=0.7)
ax3.set_title("Budget pub vs CA généré")
ax3.set_xlabel("Budget pub (k€)")
ax3.set_ylabel("CA (k€)")
ax3.grid(True, alpha=0.3)

fig.suptitle("Dashboard multi-panneaux", fontsize=13, fontweight="bold")
plt.show()

<details>
<summary>💡 Correction</summary>

```python
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

mois     = ["Jan","Fév","Mar","Avr","Mai","Jun",
            "Jul","Aoû","Sep","Oct","Nov","Déc"]
ca_mois  = [42,38,55,60,72,85,90,78,65,70,88,95]
np.random.seed(5)
ventes_j = np.random.normal(150, 40, 100).clip(50, 300)
produits = ["Pizza","Burger","Sushi","Tacos"]
ventes_p = [320, 280, 180, 150]
budget   = np.random.randint(5, 50, 30)
ca_pub   = budget * 8 + np.random.normal(0, 30, 30)

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 2, hspace=0.5, wspace=0.35)

ax0 = fig.add_subplot(gs[0, :])
ax0.plot(mois, ca_mois, color="steelblue", marker="o", linewidth=2)
ax0.set_title("CA mensuel"); ax0.set_ylabel("k€")
ax0.grid(True, alpha=0.3)

ax1 = fig.add_subplot(gs[1, 0])
ax1.hist(ventes_j, bins=12, color="coral", edgecolor="white")
ax1.set_title("Distribution ventes journalières")
ax1.set_xlabel("Ventes (€)")

ax2 = fig.add_subplot(gs[1, 1])
ax2.bar(produits, ventes_p, color=["steelblue","coral","seagreen","gold"])
ax2.set_title("Ventes par produit")
ax2.grid(True, axis="y", alpha=0.3)

ax3 = fig.add_subplot(gs[2, :])
ax3.scatter(budget, ca_pub, color="purple", alpha=0.7)
ax3.set_title("Budget pub vs CA généré")
ax3.set_xlabel("Budget pub (k€)"); ax3.set_ylabel("CA (k€)")
ax3.grid(True, alpha=0.3)

fig.suptitle("Dashboard multi-panneaux", fontsize=13, fontweight="bold")
plt.show()
```
</details>

---
## 📝 Partie 2 – Double axe Y : deux unités sur le même graphique

### 🔎 Le problème

Très courant en entreprise : on veut afficher **deux variables sur le même graphique** mais elles ont des **unités différentes** — par exemple les ventes (en unités) et le CA (en euros).

Si on les met sur le même axe Y, l'une écrase l'autre :
```
Ventes : [120, 200, 240, 90, 110]    → valeurs entre 90 et 240
CA (€) : [1380, 2400, 3000, 945, 1210] → valeurs entre 945 et 3000

Sur le même axe → l'axe va de 90 à 3000, et les ventes sont
écrasées dans le bas du graphique, illisibles.
```

### 🔎 Solution : `ax.twinx()`

`twinx()` crée un **second axe Y** à droite, qui partage le même axe X.

```python
fig, ax1 = plt.subplots(figsize=(10, 5))

# Axe gauche (ax1) : première série
ax1.plot(x, serie1, color="steelblue", label="Série 1")
ax1.set_ylabel("Unité 1", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

# Axe droit (ax2) : deuxième série — même axe X
ax2 = ax1.twinx()
ax2.plot(x, serie2, color="coral", linestyle="--", label="Série 2")
ax2.set_ylabel("Unité 2", color="coral")
ax2.tick_params(axis="y", labelcolor="coral")
```

> 💡 **Convention de lisibilité :** colorez le label de chaque axe Y dans la couleur de la courbe correspondante. L'œil fait immédiatement le lien.

In [ ]:
import matplotlib.pyplot as plt

jours   = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
ventes  = [120, 200, 240, 90, 110, 280, 195]   # en unités
ca      = [1380, 2400, 3000, 945, 1210, 3360, 2340]  # en euros

fig, ax1 = plt.subplots(figsize=(10, 5))

# Axe gauche : ventes (barres)
ax1.bar(jours, ventes, color="steelblue", alpha=0.6, label="Pizzas vendues")
ax1.set_ylabel("Pizzas vendues", color="steelblue", fontsize=11)
ax1.tick_params(axis="y", labelcolor="steelblue")
ax1.set_ylim(0, 350)

# Axe droit : CA (courbe)
ax2 = ax1.twinx()   # partage l'axe X, crée un axe Y à droite
ax2.plot(jours, ca, color="coral", marker="o", linewidth=2.5, label="CA (€)")
ax2.set_ylabel("Chiffre d'affaires (€)", color="coral", fontsize=11)
ax2.tick_params(axis="y", labelcolor="coral")
ax2.set_ylim(0, 4000)

# Légendes des deux axes combinées
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Ventes et CA par jour — Double axe Y", fontsize=13)
ax1.grid(True, axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

### 🧩 Exercice 2 – Double axe Y

Tracez sur le même graphique l'évolution mensuelle du **nombre de commandes** et du **panier moyen** :

```python
mois       = ["Jan","Fév","Mar","Avr","Mai","Jun"]
commandes  = [340, 280, 420, 460, 510, 580]   # en nombre
panier_moy = [45, 48, 42, 51, 55, 58]         # en euros
```

1. Axe gauche (bleu) : **barres** du nombre de commandes
2. Axe droit (orange) : **courbe** du panier moyen
3. Colorez chaque label d'axe dans la couleur de sa série
4. Combinez les deux légendes en une seule
5. Titre : `"Commandes et panier moyen — Semestre 1"`

In [ ]:
import matplotlib.pyplot as plt

mois       = ["Jan","Fév","Mar","Avr","Mai","Jun"]
commandes  = [340, 280, 420, 460, 510, 580]
panier_moy = [45, 48, 42, 51, 55, 58]

fig, ax1 = plt.subplots(figsize=(9, 5))

# TODO : axe gauche — barres commandes
ax1.bar(mois, commandes, color=..., alpha=0.6, label="Commandes")
ax1.set_ylabel("Nombre de commandes", color=...)
ax1.tick_params(axis="y", labelcolor=...)

# TODO : axe droit — courbe panier moyen
ax2 = ax1.twinx()
ax2.plot(mois, panier_moy, color=..., marker=..., linewidth=2.5, label="Panier moyen")
ax2.set_ylabel("Panier moyen (€)", color=...)
ax2.tick_params(axis="y", labelcolor=...)

# TODO : combiner les légendes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title(...)
ax1.grid(True, axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

<details>
<summary>💡 Correction</summary>

```python
import matplotlib.pyplot as plt

mois       = ["Jan","Fév","Mar","Avr","Mai","Jun"]
commandes  = [340, 280, 420, 460, 510, 580]
panier_moy = [45, 48, 42, 51, 55, 58]

fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.bar(mois, commandes, color="steelblue", alpha=0.6, label="Commandes")
ax1.set_ylabel("Nombre de commandes", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.plot(mois, panier_moy, color="orange", marker="o", linewidth=2.5, label="Panier moyen")
ax2.set_ylabel("Panier moyen (€)", color="orange")
ax2.tick_params(axis="y", labelcolor="orange")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Commandes et panier moyen — Semestre 1")
ax1.grid(True, axis="y", alpha=0.2)
plt.tight_layout()
plt.show()
# → Février est le mois le moins performant : peu de commandes ET panier faible
# → Mai/Juin : hausse conjointe des commandes ET du panier — excellent signal
```
</details>

---
## 📝 Partie 3 – Barres empilées : composition d'un total

### 🔎 Quand l'utiliser ?

Les **barres empilées** montrent à la fois le **total** et la **décomposition** de ce total par catégorie. Idéal pour répondre à : *"Comment le CA total se répartit-il entre nos produits chaque mois ?"*

```python
# On empile les barres avec le paramètre `bottom`
ax.bar(x, valeurs_A, label="Produit A")                   # première couche
ax.bar(x, valeurs_B, bottom=valeurs_A, label="Produit B") # posée au-dessus de A
ax.bar(x, valeurs_C, bottom=[a+b for a,b in zip(valeurs_A, valeurs_B)],
       label="Produit C")                                   # posée au-dessus de A+B
```

> 💡 **Avec NumPy**, le calcul du `bottom` est plus propre :
> ```python
> donnees = np.array([valeurs_A, valeurs_B, valeurs_C])  # shape (3, n_mois)
> cumul   = np.cumsum(donnees, axis=0)  # somme cumulée ligne par ligne
> # La 2e couche commence à cumul[0], la 3e à cumul[1], etc.
> ```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

trimestres = ["T1", "T2", "T3", "T4"]
pizza  = np.array([80, 95, 110, 100])
burger = np.array([60, 70,  65,  80])
sushi  = np.array([30, 35,  40,  45])
tacos  = np.array([20, 25,  30,  35])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ---- Barres empilées ----
ax1.bar(trimestres, pizza,  label="Pizza",  color="steelblue")
ax1.bar(trimestres, burger, label="Burger", color="coral",    bottom=pizza)
ax1.bar(trimestres, sushi,  label="Sushi",  color="seagreen", bottom=pizza+burger)
ax1.bar(trimestres, tacos,  label="Tacos",  color="gold",     bottom=pizza+burger+sushi)

ax1.set_title("CA par produit — barres empilées")
ax1.set_ylabel("CA (k€)")
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(True, axis="y", alpha=0.3)

# Afficher les totaux au-dessus des barres
totaux = pizza + burger + sushi + tacos
for i, tot in enumerate(totaux):
    ax1.text(i, tot + 2, str(tot), ha="center", fontsize=10, fontweight="bold")

# ---- Barres empilées en % (100% stacked) ----
donnees  = np.array([pizza, burger, sushi, tacos])
totaux_t = donnees.sum(axis=0)  # total par trimestre
pct      = donnees / totaux_t * 100  # pourcentages

cumul = np.zeros(len(trimestres))
couleurs = ["steelblue", "coral", "seagreen", "gold"]
labels   = ["Pizza", "Burger", "Sushi", "Tacos"]

for i, (serie, couleur, label) in enumerate(zip(pct, couleurs, labels)):
    ax2.bar(trimestres, serie, bottom=cumul, color=couleur, label=label)
    for j, (val, bot) in enumerate(zip(serie, cumul)):
        if val > 5:  # afficher % seulement si la part est visible
            ax2.text(j, bot + val/2, f"{val:.0f}%",
                     ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    cumul += serie

ax2.set_title("Répartition en % — barres 100% empilées")
ax2.set_ylabel("Part (%)")
ax2.set_ylim(0, 105)
ax2.legend(loc="upper right", fontsize=9)

plt.suptitle("Barres empilées : valeurs et pourcentages", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 📝 Partie 4 – Graphique en aires : évolution cumulée

### 🔎 Quand l'utiliser ?

Le graphique en aires (`fill_between` ou `stackplot`) est une **courbe dont la surface sous la courbe est colorée**. C'est une variante de la courbe qui met en valeur les volumes.

```python
# Aires empilées (comme les barres empilées mais en courbes)
ax.stackplot(x, serie1, serie2, serie3,
             labels  = ["Série 1", "Série 2", "Série 3"],
             colors  = ["steelblue", "coral", "seagreen"],
             alpha   = 0.7)
```

**Utile pour** : évolution du CA total décomposé par région, trafic web par source, etc.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mois   = ["Jan","Fév","Mar","Avr","Mai","Jun","Jul","Aoû","Sep","Oct","Nov","Déc"]
nord   = np.array([20, 18, 25, 28, 32, 38, 40, 35, 30, 32, 38, 42])
sud    = np.array([15, 14, 18, 20, 25, 30, 32, 28, 22, 24, 28, 32])
est    = np.array([10,  9, 12, 14, 16, 18, 20, 17, 14, 16, 18, 20])
ouest  = np.array([ 8,  7,  9, 10, 12, 14, 15, 12, 10, 11, 14, 16])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Aires empilées
ax1.stackplot(
    range(len(mois)), nord, sud, est, ouest,
    labels  = ["Nord", "Sud", "Est", "Ouest"],
    colors  = ["steelblue", "coral", "seagreen", "gold"],
    alpha   = 0.75
)
ax1.set_xticks(range(len(mois)))
ax1.set_xticklabels(mois, rotation=45)
ax1.set_title("CA par région — aires empilées")
ax1.set_ylabel("CA (k€)")
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(True, alpha=0.3)

# Même données en courbe simple pour comparer
total = nord + sud + est + ouest
ax2.plot(range(len(mois)), total, color="navy", linewidth=2.5, marker="o", label="Total")
ax2.fill_between(range(len(mois)), total, alpha=0.2, color="navy")
ax2.set_xticks(range(len(mois)))
ax2.set_xticklabels(mois, rotation=45)
ax2.set_title("CA total — courbe avec aire")
ax2.set_ylabel("CA (k€)")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.suptitle("Graphiques en aires", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 🧩 Exercice 3 – Tableau de bord rapport final

Produisez le tableau de bord complet d'une direction commerciale en combinant tout ce qu'on a vu dans ce notebook.

**Layout GridSpec 2×3 :**
- **Haut gauche (2 colonnes)** : courbe CA mensuel + ligne objectif + annotations pic/creux
- **Haut droit** : camembert répartition CA par région
- **Bas gauche** : barres empilées (CA par produit sur 4 trimestres)
- **Bas milieu** : double axe Y (commandes + panier moyen sur 6 mois)
- **Bas droit** : texte KPIs (CA total, meilleur mois, taux croissance)

```python
# Données
mois    = ["Jan","Fév","Mar","Avr","Mai","Jun","Jul","Aoû","Sep","Oct","Nov","Déc"]
ca      = [42,38,55,60,72,85,90,78,65,70,88,95]
obj     = 70
regions = ["Nord","Sud","Est","Ouest"]
parts   = [35, 28, 22, 15]
tri     = ["T1","T2","T3","T4"]
pizza   = [80, 95, 110, 100]
burger  = [60, 70,  65,  80]
mois6   = ["Jan","Fév","Mar","Avr","Mai","Jun"]
cmd     = [340, 280, 420, 460, 510, 580]
panier  = [45, 48, 42, 51, 55, 58]
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

mois    = ["Jan","Fév","Mar","Avr","Mai","Jun","Jul","Aoû","Sep","Oct","Nov","Déc"]
ca      = [42,38,55,60,72,85,90,78,65,70,88,95]
obj     = 70
regions = ["Nord","Sud","Est","Ouest"]
parts   = [35, 28, 22, 15]
tri     = ["T1","T2","T3","T4"]
pizza   = np.array([80, 95, 110, 100])
burger  = np.array([60, 70,  65,  80])
mois6   = ["Jan","Fév","Mar","Avr","Mai","Jun"]
cmd     = [340, 280, 420, 460, 510, 580]
panier  = [45, 48, 42, 51, 55, 58]

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

# --- Haut gauche : courbe CA + objectif + annotations ---
ax0 = fig.add_subplot(gs[0, 0:2])
ax0.plot(mois, ca, color="steelblue", marker="o", linewidth=2)
ax0.axhline(obj, color="red", linestyle="--", label=f"Objectif {obj}k€")
idx_max = ca.index(max(ca))
ax0.annotate(f"Pic : {max(ca)}k€",
    xy=(idx_max, max(ca)), xytext=(idx_max-2, max(ca)+5),
    arrowprops=dict(arrowstyle="->", color="red"),
    color="red", fontsize=9)
ax0.set_title("CA mensuel vs objectif")
ax0.set_ylabel("k€")
ax0.legend(fontsize=9)
ax0.tick_params(axis="x", rotation=45)
ax0.grid(True, alpha=0.3)

# --- Haut droit : camembert ---
ax1 = fig.add_subplot(gs[0, 2])
ax1.pie(parts, labels=regions, autopct="%1.0f%%", startangle=90,
        colors=["steelblue","coral","seagreen","gold"])
ax1.set_title("CA par région")

# --- Bas gauche : barres empilées ---
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(tri, pizza,  label="Pizza",  color="steelblue")
ax2.bar(tri, burger, label="Burger", color="coral", bottom=pizza)
ax2.set_title("Ventes par produit")
ax2.set_ylabel("k€")
ax2.legend(fontsize=8)
ax2.grid(True, axis="y", alpha=0.3)

# --- Bas milieu : double axe Y ---
ax3a = fig.add_subplot(gs[1, 1])
ax3a.bar(mois6, cmd, color="steelblue", alpha=0.6, label="Commandes")
ax3a.set_ylabel("Commandes", color="steelblue", fontsize=9)
ax3a.tick_params(axis="y", labelcolor="steelblue")
ax3b = ax3a.twinx()
ax3b.plot(mois6, panier, color="orange", marker="o", linewidth=2, label="Panier")
ax3b.set_ylabel("Panier (€)", color="orange", fontsize=9)
ax3b.tick_params(axis="y", labelcolor="orange")
l1,lbl1 = ax3a.get_legend_handles_labels()
l2,lbl2 = ax3b.get_legend_handles_labels()
ax3a.legend(l1+l2, lbl1+lbl2, fontsize=8, loc="upper left")
ax3a.set_title("Commandes & Panier moyen")
ax3a.tick_params(axis="x", rotation=45)

# --- Bas droit : KPIs textuels ---
ax4 = fig.add_subplot(gs[1, 2])
ax4.axis("off")
croissance = (ca[-1] - ca[0]) / ca[0] * 100
ax4.text(0.5, 0.85, "📊 KPIs Annuels",    ha="center", fontsize=12, fontweight="bold")
ax4.text(0.5, 0.65, f"CA Total : {sum(ca)} k€", ha="center", fontsize=11, color="steelblue")
ax4.text(0.5, 0.48, f"Meilleur mois : {mois[idx_max]}", ha="center", fontsize=10)
ax4.text(0.5, 0.32, f"Mois > objectif : {sum(1 for v in ca if v>=obj)}/12",
         ha="center", fontsize=10, color="seagreen")
ax4.text(0.5, 0.15, f"Croissance Jan→Déc : {croissance:+.0f}%",
         ha="center", fontsize=10, color="seagreen" if croissance > 0 else "crimson")

fig.suptitle("Tableau de bord Direction Commerciale",
             fontsize=15, fontweight="bold", y=1.01)
plt.savefig("tableau_de_bord.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ tableau_de_bord.png sauvegardé")

<details>
<summary>💡 Correction</summary>

Le code ci-dessus **est** la correction — l'exercice guide étape par étape. Vérifiez que votre tableau de bord affiche bien les 5 panneaux avec les bonnes données et qu'il est sauvegardé en PNG.

**Points clés à vérifier :**
- Le layout `GridSpec(2, 3)` avec `gs[0, 0:2]` pour le panneau large
- `ax.twinx()` partage bien l'axe X
- `ax.axis("off")` supprime les axes pour le panneau texte
- `plt.savefig()` avant `plt.show()`
</details>

---
## ✅ Récapitulatif

| Concept | Code | Ce qu'il faut retenir |
|---------|------|-----------------------|
| **GridSpec** | `gs = gridspec.GridSpec(n_lignes, n_cols)` | Panneaux de tailles différentes |
| **Panneau large** | `fig.add_subplot(gs[0, 0:2])` | Slice comme NumPy |
| **Espacement** | `GridSpec(..., hspace=0.4, wspace=0.3)` | Entre lignes et colonnes |
| **Double axe Y** | `ax2 = ax1.twinx()` | Deux unités sur le même graphique |
| **Couleur axe** | `ax.tick_params(axis="y", labelcolor="coral")` | Relier visuellement axe et courbe |
| **Combiner légendes** | `lines1+lines2, labels1+labels2` | Une seule légende pour deux axes |
| **Barres empilées** | `ax.bar(x, y2, bottom=y1)` | Composition d'un total |
| **Aires empilées** | `ax.stackplot(x, s1, s2, s3)` | Évolution de la composition |
| **Panneau texte** | `ax.axis("off")` + `ax.text(x, y, ...)` | KPIs, commentaires |

**Ce que vous savez faire maintenant :**
- Créer n'importe quel layout de tableau de bord avec GridSpec
- Comparer deux variables d'unités différentes avec twinx
- Visualiser la décomposition d'un total avec les barres empilées
- Combiner tous les types de graphiques dans un rapport prêt à présenter

---
📘 **Prochain module → Seaborn : visualisation statistique avancée**